# Q3 — Hindi Spelling Correctness Detection (~1.77 Lakh Words)

**Dataset**: ~1,77,000 unique words from the JoshTalks human-transcribed Hindi dataset  
**Source**: [Q3 Google Sheet](https://docs.google.com/spreadsheets/d/17DwCAx6Tym5Nt7eOni848np9meR-TIj7uULMtYcgQaw/edit)

**Transcription guideline**: English words in Devanagari (e.g., `कंप्यूटर`) are **correct** — they should NOT be flagged as misspellings.

**Output required**:
- (a) Correct vs incorrect classification for every word
- (b) Confidence: high / medium / low + brief reason
- (c) Low-confidence review: accuracy + failure pattern analysis
- (d) Unreliable categories + why

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from data_loader import Q3WordListLoader
from spelling_checker import HindiSpellingChecker

print('Imports OK ✓')

---
## 3(a). Approach

| Layer | Method | Handles |
|---|---|---|
| 1 | **Dictionary exact match** | Common correct words | 
| 2 | **Levenshtein distance ≤ 2** | Likely misspellings near known words |
| 3 | **Devanagari phonetic encoding** | Sound-alike confusions (ण↔न, श↔ष, ँ↔ं) |
| 4 | **English-in-Devanagari heuristic** | `कंप्यूटर`, `इंटरव्यू` etc. → marked correct |

**Why not ML?** With 1.77L words and no labelled training set, a rule-based  
approach is more reliable and interpretable. A morphological analyser (IndicNLP)  
would be the production upgrade.

In [ ]:
# ── Load real Q3 word list from Google Sheets ─────────────────────────────
q3_loader = Q3WordListLoader(cache_path='../data/processed/q3_words.csv')
all_words = q3_loader.load()

print(f'Total unique words loaded: {len(all_words):,}')
print('Sample words:')
import random; random.seed(42)
print(random.sample(all_words, min(10, len(all_words))))

In [ ]:
# ── Initialise spelling checker ───────────────────────────────────────────
# Provide a Hindi dictionary if you have one (one word per line):
# DICT_PATH = '../data/raw/hindi_dictionary.txt'
DICT_PATH = None  # Uses built-in seed + fallback heuristics

checker = HindiSpellingChecker(
    dictionary_path=DICT_PATH,
    max_edit_distance=2,
    use_phonetics=True,
)
print(f'Dictionary size: {len(checker.dictionary):,} words')

In [ ]:
# ── Quick sanity check on known examples ─────────────────────────────────
spot_check = [
    # Correct Hindi words
    'नमस्ते', 'भारत', 'पानी', 'खेती', 'बाड़ी', 'मौनता', 'रक्षाबंधन',
    # English loanwords in Devanagari (should be CORRECT per guidelines)
    'इंटरव्यू', 'जॉब', 'मोबाइल', 'स्कूल', 'कंप्यूटर',
    # Likely misspellings
    'नमसते', 'भारथ', 'खेतिबाड़ी',
]

rows = []
for w in spot_check:
    r = checker.check_word(w)
    rows.append({'Word': r.word, 'Label': r.label, 'Conf': r.confidence,
                 'Score': r.confidence_score, 'Nearest': r.nearest_match})
display(pd.DataFrame(rows))

---
## 3(a–b). Full Batch Classification

In [ ]:
# Classify all words
# ⚠️  With 1.77L words this takes ~5–10 min on CPU (lru_cache accelerates repeats)
print(f'Classifying {len(all_words):,} words …')
results = checker.check_batch(all_words)

# Save full results
checker.save_results(results, '../outputs/spelling_results.csv')

df = pd.read_csv('../outputs/spelling_results.csv')
n_correct   = (df['label'] == 'correct').sum()
n_incorrect = (df['label'] == 'incorrect').sum()

print(f'\n=== 3(a) Classification Summary ===')
print(f'Correctly spelled unique words:   {n_correct:,}  ({100*n_correct/len(df):.1f}%)')
print(f'Incorrectly spelled unique words: {n_incorrect:,}  ({100*n_incorrect/len(df):.1f}%)')
print(f'Total unique words classified:    {len(df):,}')

In [ ]:
print('\n=== 3(b) Confidence Distribution ===')
conf_counts = df.groupby(['label', 'confidence']).size().unstack(fill_value=0)
display(conf_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

label_c = df['label'].value_counts()
axes[0].pie(label_c, labels=label_c.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c'], startangle=90)
axes[0].set_title('Spelling Classification')

axes[1].hist(df['confidence_score'], bins=30, color='#3498db', edgecolor='white')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence Score Distribution')

plt.tight_layout()
plt.savefig('../outputs/q3_spelling_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3(c). Low-Confidence Review (40–50 words)

In [ ]:
analysis = checker.low_confidence_analysis(results, n_samples=50)

print(f'Total low-confidence words: {analysis["total_low_confidence"]:,}')
print(f'Sample reviewed:            {analysis["sample_size"]}')
print(f'Failure patterns observed:')
for pat, cnt in analysis['failure_patterns'].items():
    if cnt > 0:
        print(f'  {pat}: {cnt}')

# Manual review: show sample with predicted label and reasoning
low_conf_words = [r for r in results if r.confidence == 'low'][:50]
lc_rows = [{
    'Word': r.word,
    'Predicted': r.label,
    'Score': r.confidence_score,
    'Nearest': r.nearest_match,
    'Reason': r.reason[:60] + '...' if len(r.reason) > 60 else r.reason,
} for r in low_conf_words[:15]]
display(pd.DataFrame(lc_rows))

In [ ]:
# Accuracy estimate on labelled ground truth (we know the expected labels)
ground_truth = [
    # Correct Hindi words
    ('नमस्ते','correct'), ('भारत','correct'), ('रक्षाबंधन','correct'),
    ('मौनता','correct'), ('खेती','correct'), ('बाड़ी','correct'),
    # English loanwords (correct per guidelines)
    ('इंटरव्यू','correct'), ('जॉब','correct'), ('मोबाइल','correct'),
    ('स्कूल','correct'), ('कंप्यूटर','correct'),
    # Misspellings
    ('नमसते','incorrect'), ('भारथ','incorrect'), ('पाणि','incorrect'),
    ('घार','incorrect'), ('अच्चा','incorrect'),
]

hits = 0
gt_rows = []
for word, true_label in ground_truth:
    r = checker.check_word(word)
    match = r.label == true_label
    hits += int(match)
    gt_rows.append({'Word': word, 'True': true_label, 'Predicted': r.label,
                    'Conf': r.confidence, 'Correct?': '✓' if match else '✗'})

accuracy = hits / len(ground_truth)
print(f'\n3(c) Accuracy on {len(ground_truth)} labelled words: {accuracy:.2%}')
print(f'Correct: {hits} / {len(ground_truth)}')
display(pd.DataFrame(gt_rows))

---
## 3(d). Unreliable Categories

| Category | Why Unreliable |
|---|---|
| **English loanwords in Devanagari** | `कंप्यूटर`, `इंटरव्यू` don't appear in Hindi dictionaries → falsely flagged as incorrect. Need a Devanagari loanword lexicon. |
| **Inflected/compound words** | `रक्षाबंधन`, `खेतीबाड़ी` — compounds may not appear in base dictionary. Morphological decomposition needed. |
| **Very short words (≤2 chars)** | Ambiguous without context (e.g., `मैं`, `वो`, `हूँ` look strange to edit-distance but are valid). |
| **Dialectal/regional variants** | Words from Bhojpuri, Rajasthani, etc. mixed into Hindi ASR are not in standard dictionaries. |

In [ ]:
# Demonstrate unreliable categories with real examples
unreliable_examples = [
    # English loanwords
    'कंप्यूटर', 'इंटरव्यू', 'जॉब', 'प्रॉब्लम', 'मैनेजर',
    # Compounds
    'रक्षाबंधन', 'खेतीबाड़ी', 'लोकतंत्र', 'सर्वशक्तिमान',
    # Short
    'में', 'हूँ', 'को', 'से', 'पर',
]

rows = []
for w in unreliable_examples:
    r = checker.check_word(w)
    rows.append({'Word': w, 'Label': r.label, 'Confidence': r.confidence,
                 'Score': round(r.confidence_score, 2), 'Nearest': r.nearest_match})

display(pd.DataFrame(rows))
print('\nNote: English loanwords in Devanagari should all be labelled "correct"')
print('but will often be low-confidence / incorrect until a loanword lexicon is added.')